# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [1]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [2]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/agedpayables"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 5

--- AgedPayables_20260207170508.parquet ---
  Registros: 1
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260207170624.parquet ---
  Registros: 1
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260325043000.parquet ---
  Registros: 10
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260325043100.parquet ---
  Registros: 10
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']

--- AgedPayables_20260325214230.parquet ---
  Registros: 1
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']



import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [3]:
from pyspark.sql.functions import from_json, schema_of_json, col, explode
from pyspark.sql.types import ArrayType, StructType

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema))

        # Se o resultado for um array, explodir primeiro
        parsed_type = df_temp.schema["parsed"].dataType
        if isinstance(parsed_type, ArrayType):
            df_temp = df_temp.withColumn("parsed", explode(col("parsed")))

        # Manter colunas originais (sem a coluna JSON bruta) + colunas expandidas do JSON
        colunas_originais = [c for c in df.columns if c != col_json]
        df_temp = df_temp.select(*colunas_originais, col("parsed.*"))

        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 5

Comparando: AgedPayables_20260207170508.parquet vs AgedPayables_20260207170624.parquet
  ✅ Mesmas colunas (3 colunas)
  ✅ Tipos de dados idênticos

Comparando: AgedPayables_20260207170508.parquet vs AgedPayables_20260325043000.parquet
  ✅ Mesmas colunas (3 colunas)
  ❌ Tipos de dados diferem:
     Coluna 'Row': array<struct<ColData:array<struct<id:string,value:string>>,Summary:struct<ColData:array<struct<value:string>>>,group:string,type:string>> → array<struct<Header:struct<ColData:array<struct<value:string>>>,Rows:struct<Row:array<struct<ColData:array<struct<value:string>>>>>,Summary:struct<ColData:array<struct<value:string>>>>>

Comparando: AgedPayables_20260207170508.parquet vs AgedPayables_20260325043100.parquet
  ✅ Mesmas colunas (3 colunas)
  ❌ Tipos de dados diferem:
     Coluna 'Row': array<struct<ColData:array<struct<id:string,value:string>>,Summary:struct<ColData:array<struct<value:string>>>,group:string,type:string>> → array<struct<Header:struct<ColDat

In [4]:
from pyspark.sql.functions import from_json, schema_of_json, col, explode_outer
from pyspark.sql.types import StructType, StringType, ArrayType
from functools import reduce

def simplify_value_only_columns(df):
    """Se uma coluna for um struct com apenas a chave 'value', substitui pelo valor direto."""
    for field in df.schema.fields:
        if isinstance(field.dataType, StructType):
            nomes_campos = [f.name for f in field.dataType.fields]
            if nomes_campos == ["value"]:
                df = df.withColumn(field.name, col(f"{field.name}.value"))
    return df

def flatten_df(df, sep="_"):
    """Achata recursivamente colunas StructType e explode ArrayType."""
    df = simplify_value_only_columns(df)
    changed = True
    while changed:
        changed = False

        # Explodir colunas array (usando explode_outer para preservar linhas com arrays nulos)
        for field in df.schema.fields:
            if isinstance(field.dataType, ArrayType):
                df = df.withColumn(field.name, explode_outer(col(field.name)))
                changed = True

        # Achatar colunas struct
        novas_colunas = []
        for field in df.schema.fields:
            if isinstance(field.dataType, StructType):
                nomes_campos = [f.name for f in field.dataType.fields]
                if nomes_campos == ["value"]:
                    novas_colunas.append(df[field.name]["value"].alias(field.name))
                else:
                    changed = True
                    for sub in field.dataType.fields:
                        novo_nome = f"{field.name}{sep}{sub.name}"
                        novas_colunas.append(df[field.name][sub.name].alias(novo_nome))
            else:
                novas_colunas.append(df[field.name])
        df = df.select(novas_colunas)
        df = simplify_value_only_columns(df)
    return df

def parse_nested_json(df, sep="_", max_depth=5):
    """Detecta e parseia colunas string que contêm JSON, depois achata."""
    for _ in range(max_depth):
        str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        if not str_cols:
            break

        # Coletar amostras não-nulas de cada coluna string
        cols_json = []
        for c in str_cols:
            row = df.select(c).filter(col(c).isNotNull()).first()
            if row:
                val = row[0]
                if val and isinstance(val, str) and val.strip() and val.strip()[0] in ('{', '['):
                    cols_json.append((c, val))

        if not cols_json:
            break

        for c, val in cols_json:
            schema = schema_of_json(val)
            df = df.withColumn(c, from_json(df[c], schema))

        df = flatten_df(df, sep)
    return df

# Processar cada arquivo: parsear TODAS as colunas JSON em sequência no mesmo DataFrame
dfs_processados = []

for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"⚠️ Pulando {nome}: sem dados JSON para processar ({info['df'].columns})")
        continue

    df = info["df"]

    # Parsear todas as colunas JSON de uma vez, em sequência no mesmo df
    for col_json in info["colunas_json"]:
        amostra_row = df.select(col_json).filter(col(col_json).isNotNull()).first()
        if not amostra_row:
            continue
        amostra = amostra_row[0]
        json_schema = schema_of_json(amostra)

        # Substituir a coluna JSON pela versão parseada
        df = df.withColumn(col_json, from_json(col(col_json), json_schema))

    # Achatar tudo (structs, arrays, value-only) de uma vez
    df = flatten_df(df)

    # Parsear sub-JSONs que possam ter surgido como strings após achatamento
    df = parse_nested_json(df)
    df = simplify_value_only_columns(df)

    info["df_parsed"] = df
    dfs_processados.append(df)
    print(f"✅ {nome}: {df.count()} registros, {len(df.columns)} colunas")

# Unir todos os DataFrames e remover duplicados
df_final = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_processados)
df_final = df_final.dropDuplicates()

print(f"\nTotal de registros (sem duplicados): {df_final.count()}")
print(f"Total de colunas: {len(df_final.columns)}")
df_final.printSchema()

✅ AgedPayables_20260207170508.parquet: 588 registros, 16 colunas
✅ AgedPayables_20260207170624.parquet: 588 registros, 16 colunas
✅ AgedPayables_20260325043000.parquet: 160 registros, 11 colunas
✅ AgedPayables_20260325043100.parquet: 160 registros, 11 colunas
✅ AgedPayables_20260325214230.parquet: 588 registros, 16 colunas

Total de registros (sem duplicados): 860
Total de colunas: 18
root
 |-- Header_Currency: string (nullable = true)
 |-- Header_EndPeriod: string (nullable = true)
 |-- Header_Option_Name: string (nullable = true)
 |-- Header_Option_Value: string (nullable = true)
 |-- Header_ReportName: string (nullable = true)
 |-- Header_SummarizeColumnsBy: string (nullable = true)
 |-- Header_Time: string (nullable = true)
 |-- Columns_Column_ColTitle: string (nullable = true)
 |-- Columns_Column_ColType: string (nullable = true)
 |-- Columns_Column_MetaData_Name: string (nullable = true)
 |-- Columns_Column_MetaData_Value: string (nullable = true)
 |-- Rows_Row_ColData_id: string

In [5]:
from IPython.display import display

# Mostrar tabela única consolidada
display(df_final.toPandas())

,Header_Currency,Header_EndPeriod,Header_Option_Name,Header_Option_Value,Header_ReportName,Header_SummarizeColumnsBy,Header_Time,Columns_Column_ColTitle,Columns_Column_ColType,Columns_Column_MetaData_Name,Columns_Column_MetaData_Value,Rows_Row_ColData_id,Rows_Row_ColData_value,Rows_Row_Summary_ColData,Rows_Row_group,Rows_Row_type,Rows_Row_Header_ColData,Rows_Row_Rows_Row_ColData
0,USD,2026-02-07,report_date,2026-02-07,AgedPayables,Total,2026-02-07T12:05:31-08:00,1 - 30,Money,ColKey,0,None,,None,None,None,None,None
1,USD,2026-02-07,NoReportData,false,AgedPayables,Total,2026-02-07T12:05:31-08:00,Current,Money,ColKey,current,None,241.23,None,None,None,None,None
2,USD,2026-02-07,NoReportData,false,AgedPayables,Total,2026-02-07T12:05:31-08:00,Current,Money,ColKey,current,None,None,761.23,GrandTotal,Section,None,None
3,USD,2026-02-07,report_date,2026-02-07,AgedPayables,Total,2026-02-07T12:05:31-08:00,Current,Money,ColKey,current,48,PG&E,None,None,None,None,None
4,USD,2026-02-07,report_date,2026-02-07,AgedPayables,Total,2026-02-07T12:05:31-08:00,1 - 30,Money,ColKey,0,46,Norton Lumber and Building Materials,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
855,USD,2026-03-25,NoReportData,false,AgedPayables,Total,2026-03-25T14:42:30-07:00,1 - 30,Money,ColKey,0,None,None,1602.67,GrandTotal,Section,None,None
856,USD,2026-03-25,NoReportData,false,AgedPayables,Total,2026-03-25T14:42:30-07:00,,Vendor,None,None,48,PG&E,None,None,None,None,None
857,USD,2026-03-25,NoReportData,false,AgedPayables,Total,2026-03-25T14:42:30-07:00,91 and over,Money,ColKey,3,48,PG&E,None,None,None,None,None
858,USD,2026-03-25,report_date,2026-03-25,AgedPayables,Total,2026-03-25T14:42:30-07:00,1 - 30,Money,ColKey,0,49,Robertson & Associates,None,None,None,None,None


In [6]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
